## Spliting for 3 Tesla scans only:

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
from pathlib import Path

meta = pd.read_csv(
    "/home/enooo/Documents/Master/Early_Detection_AD/01_data/csv/master_metadata_3t.csv"
)

# ── Filter 1: keep only 3T scans ──────────────────────────────────────────────
meta_3t = meta[meta['mri_field_str'] == 3.0].copy()

# ── Filter 2: drop MCI class (ADNI-1 artifact, mixed protocol) ───────────────
meta_3t = meta_3t[meta_3t['class'] != 'MCI'].copy()

# ── One row per subject (subject_id + class) ──────────────────────────────────
subjects = meta_3t[['subject_id', 'class']].drop_duplicates()

print("Subjects after filtering (3T only, no MCI):", len(subjects))
print(subjects['class'].value_counts())

Subjects after filtering (3T only, no MCI): 462
class
EMCI    169
CN      124
LMCI     85
AD       84
Name: count, dtype: int64


## 1st split train 70, temp 30

In [2]:
train_subj, temp_subj = train_test_split(
    subjects,
    test_size=0.30,
    stratify=subjects['class'],
    random_state=42
)

## 2nd split: 15 val, 15 test

In [3]:
val_subj, test_subj = train_test_split(
    temp_subj,
    test_size=0.50,
    stratify=temp_subj['class'],
    random_state=42
)

In [4]:
print("\nTRAIN")
print(train_subj['class'].value_counts())

print("\nVAL")
print(val_subj['class'].value_counts())

print("\nTEST")
print(test_subj['class'].value_counts())


TRAIN
class
EMCI    118
CN       87
LMCI     59
AD       59
Name: count, dtype: int64

VAL
class
EMCI    25
CN      18
AD      13
LMCI    13
Name: count, dtype: int64

TEST
class
EMCI    26
CN      19
LMCI    13
AD      12
Name: count, dtype: int64


# check data leakage

In [5]:
assert set(train_subj.subject_id).isdisjoint(val_subj.subject_id)
assert set(train_subj.subject_id).isdisjoint(test_subj.subject_id)
assert set(val_subj.subject_id).isdisjoint(test_subj.subject_id)

print("✓ No subject leakage")

✓ No subject leakage


In [6]:
split_dir = Path(
    "/home/enooo/Documents/Master/Early_Detection_AD/01_data/splits"
)

split_dir.mkdir(
    parents=True,
    exist_ok=True
)

train_subj.to_csv(
    split_dir / "train_subjects.csv",
    index=False
)

val_subj.to_csv(
    split_dir / "val_subjects.csv",
    index=False
)

test_subj.to_csv(
    split_dir / "test_subjects.csv",
    index=False
)

print("Saved split files.")

Saved split files.


In [7]:
# ── Subject-level split summary ───────────────────────────────────────────────
summary = pd.concat([
    train_subj.assign(split='train'),
    val_subj.assign(split='val'),
    test_subj.assign(split='test')
])

print("=== Subject-level split ===")
print(pd.crosstab(summary['class'], summary['split']))

# ── Scan-level split (all visits per subject) ─────────────────────────────────
# Merge split assignments back onto the full 3T/no-MCI scan DataFrame
scans_with_split = meta_3t.merge(
    summary[['subject_id', 'split']],
    on='subject_id',
    how='inner'
)

print("\n=== Scan-level split (visits included) ===")
print(pd.crosstab(scans_with_split['class'], scans_with_split['split']))

print(f"\nTotal scans : {len(scans_with_split)}")
print(f"Total subjects: {len(subjects)}")

=== Subject-level split ===
split  test  train  val
class                  
AD       12     59   13
CN       19     87   18
EMCI     26    118   25
LMCI     13     59   13

=== Scan-level split (visits included) ===
split  test  train  val
class                  
AD       31    175   32
CN       71    298   67
EMCI     78    484   96
LMCI     42    211   41

Total scans : 1626
Total subjects: 462
